In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("bajaj_finance_policy_prose_v1.pdf")
raw_docs = loader.load()

/Users/rahultiwari/Documents/02_Freelancing/coding_ninja_fresh/dummy-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0,
                                          separators=["\n\n", "\n", " "])

chunks = splitter.split_documents(raw_docs)
print()
print(f"✅ Corpus built: {len(chunks)} documents")


✅ Corpus built: 43 documents


In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")


In [5]:
# pip install langchain-pinecone
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

pc = Pinecone()


In [8]:

if "bajajbot-policy" not in pc.list_indexes().names():
    pc.create_index(
        name="bajajbot-policy",
        dimension=1536,       # must match embedding model
        metric="cosine",       # cosine similarity for text
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )




In [9]:
vectorstore = PineconeVectorStore.from_documents(chunks,
                                                  embeddings, 
                                                  index_name="bajajbot-policy")

In [11]:
index = pc.Index("bajajbot-policy")

stats = index.describe_index_stats()
stats

{'dimension': 1536,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 195}},
 'total_vector_count': 195,
 'vector_type': 'dense'}

In [15]:
result = index.fetch(ids=['66555af0-3539-4182-870f-2e8d3cdee9c4'])

In [16]:
result

FetchResponse(namespace='', vectors={'66555af0-3539-4182-870f-2e8d3cdee9c4': Vector(id='66555af0-3539-4182-870f-2e8d3cdee9c4', values=[-0.0223388672, 0.0426635742, 0.0766601562, -0.0254058838, -0.00584793091, 0.032623291, 0.0080871582, 0.0398864746, -0.00880432129, 0.0424194336, 0.0185852051, -0.0882568359, -0.0176544189, -0.0110549927, 0.0246734619, 0.0347595215, -0.0117645264, -0.00193691254, -0.0360412598, 0.0472106934, 0.0305023193, 0.0588989258, 0.00194358826, 0.00951385498, 0.00930786133, -0.00904846191, -0.0300598145, 0.0130615234, 0.0307159424, -0.061126709, 0.0657959, -0.00343894958, 0.00592803955, 0.0111083984, -0.00385856628, 0.0103759766, 0.0052986145, -0.00796508789, 0.0859375, -0.022354126, -0.0175170898, -0.0350646973, -0.0137786865, -0.035736084, 0.0146179199, 0.00226211548, -0.0288238525, -0.00995636, -0.0160827637, 0.0471191406, 0.00193023682, -0.020111084, -0.00309944153, -0.0176086426, -0.0271759033, -0.0208740234, -0.0215759277, 0.0190734863, 0.00363349915, -0.0254

In [25]:
print(result.vectors['66555af0-3539-4182-870f-2e8d3cdee9c4'])

Vector(id='66555af0-3539-4182-870f-2e8d3cdee9c4', values=[-0.0223388672, 0.0426635742, 0.0766601562, -0.0254058838, -0.00584793091, 0.032623291, 0.0080871582, 0.0398864746, -0.00880432129, 0.0424194336, 0.0185852051, -0.0882568359, -0.0176544189, -0.0110549927, 0.0246734619, 0.0347595215, -0.0117645264, -0.00193691254, -0.0360412598, 0.0472106934, 0.0305023193, 0.0588989258, 0.00194358826, 0.00951385498, 0.00930786133, -0.00904846191, -0.0300598145, 0.0130615234, 0.0307159424, -0.061126709, 0.0657959, -0.00343894958, 0.00592803955, 0.0111083984, -0.00385856628, 0.0103759766, 0.0052986145, -0.00796508789, 0.0859375, -0.022354126, -0.0175170898, -0.0350646973, -0.0137786865, -0.035736084, 0.0146179199, 0.00226211548, -0.0288238525, -0.00995636, -0.0160827637, 0.0471191406, 0.00193023682, -0.020111084, -0.00309944153, -0.0176086426, -0.0271759033, -0.0208740234, -0.0215759277, 0.0190734863, 0.00363349915, -0.0254821777, 0.0184326172, 0.027557373, -0.018447876, -0.0177764893, -0.0019912719

In [30]:
result.vectors['66555af0-3539-4182-870f-2e8d3cdee9c4'].metadata['text']

'sectors, extending to five years for manufacturing businesses.\n6.2 Sector-Specific Rules\nBajaj Finance applies different eligibility criteria depending on the sector in which the applicant operates.\nThis reflects the varying risk profiles and cash flow patterns across industries.\nTrading and retail businesses require a minimum vintage of three years and may borrow up to Rs 50\nlakhs without collateral. GST returns are mandatory for this sector. For loan amounts above Rs 25 lakhs\nin this segment, a stock audit is required before disbursement.\nService businesses including IT firms and consulting companies also require three years of vintage and\nare eligible for up to Rs 50 lakhs unsecured. For loans above Rs 20 lakhs in this segment, client\ncontracts must be provided as supplementary income proof.\nManufacturing businesses face the strictest requirements: a minimum of five years of vintage, a cap of'

In [32]:
query = embeddings.embed_query("Bajaj Finance applies different eligibility criteria depending on the sector in which the applicant operates.")

result = index.query(
    vector = query,
    top_k = 3,
    include_metadata=True
)
# result.matches

In [33]:
result.matches

[{'id': 'bajaj_finance_policy_prose_v1_p9_2',
  'metadata': {'ingested_on': '2026-05-30',
               'page': 9.0,
               'source_file': 'bajaj_finance_policy_prose_v1.pdf',
               'text': 'sectors, extending to five years for manufacturing '
                       'businesses.\n'
                       '6.2 Sector-Specific Rules\n'
                       'Bajaj Finance applies different eligibility criteria '
                       'depending on the sector in which the applicant '
                       'operates.\n'
                       'This reflects the varying risk profiles and cash flow '
                       'patterns across industries.\n'
                       'Trading and retail businesses require a minimum vintage '
                       'of three years and may borrow up to Rs 50\n'
                       'lakhs without collateral. GST returns are mandatory for '
                       'this sector. For loan amounts above Rs 25 lakhs'},
  'score': 0.6